# Detecção de Fraude em Transações de Cartão de Crédito

Este notebook segue o pipeline completo proposto no desafio: exploração dos dados,
tratamento do desbalanceamento, treinamento e comparação de modelos, e explicação
das decisões do modelo (feature importance + SHAP).

**Dataset:** *Credit Card Fraud Detection* (transações de cartões europeus, set/2013),
284.807 transações, das quais 492 são fraude (~0,17%). Colunas: `Time`, `Amount`,
`Class` (0 = normal, 1 = fraude) e `V1`...`V28` (variáveis anonimizadas via PCA).

> ⚠️ O dataset **não** é versionado neste repositório — ele é baixado por link em
> tempo de execução (célula abaixo). Se a Aula 1 do seu curso indicar um link
> diferente (ex.: Kaggle), basta trocar a variável `DATA_URL`.


## 1. Configuração e carregamento dos dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, precision_score, recall_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost não está instalado. Rode: pip install xgboost")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")


In [ ]:
# Link público do dataset (mirror sem necessidade de login).
# Se a Aula 1 do seu curso apontar outro link (ex.: Kaggle), troque aqui.
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(DATA_URL)
print("Shape do dataset:", df.shape)
df.head()


## 2. Exploração (EDA)

O primeiro passo é entender a estrutura da base e medir o desbalanceamento entre
as classes — é esse desbalanceamento que torna a acurácia uma métrica enganosa
neste problema.

In [ ]:
df.info()


In [ ]:
df.isna().sum().sum()  # checagem rápida de valores ausentes


In [ ]:
contagem = df['Class'].value_counts()
proporcao = df['Class'].value_counts(normalize=True) * 100

print(contagem)
print()
print(proporcao.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=contagem.index, y=contagem.values, ax=ax)
ax.set_xticklabels(['Normal (0)', 'Fraude (1)'])
ax.set_ylabel('Nº de transações')
ax.set_title('Distribuição das classes (escala linear)')
for i, v in enumerate(contagem.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.show()


In [ ]:
# Estatísticas descritivas de Amount e Time, por classe
df.groupby('Class')[['Amount', 'Time']].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df.loc[df.Class == 0, 'Amount'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Amount — transações normais')
sns.histplot(df.loc[df.Class == 1, 'Amount'], bins=50, ax=axes[1], color='indianred')
axes[1].set_title('Amount — transações fraudulentas')
plt.tight_layout()
plt.show()


## 3. Preparação dos dados

- Criamos `log_amount` (log1p do valor) para reduzir a assimetria da distribuição de `Amount`.
- Padronizamos `Amount`, `log_amount` e `Time` com `StandardScaler` (as colunas `V1`...`V28`
  já vêm padronizadas pelo PCA original).
- Separação treino/teste com `stratify=y`, preservando a proporção de fraudes nos dois
  conjuntos — essencial num problema tão desbalanceado.

In [ ]:
data = df.copy()
data['log_amount'] = np.log1p(data['Amount'])

features_to_scale = ['Amount', 'log_amount', 'Time']
scaler = StandardScaler()
data[features_to_scale] = scaler.fit_transform(data[features_to_scale])

X = data.drop(columns=['Class'])
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print("Treino:", X_train.shape, "  Proporção de fraude:", y_train.mean().round(5))
print("Teste: ", X_test.shape, "  Proporção de fraude:", y_test.mean().round(5))


## 4. Treinamento dos modelos

Começamos por uma **Regressão Logística** como baseline (simples e interpretável) e
comparamos com **Random Forest** e **XGBoost**. Em todos, usamos o peso das classes
(`class_weight='balanced'` / `scale_pos_weight`) para compensar o desbalanceamento
sem precisar duplicar ou descartar dados na primeira rodada.

In [ ]:
log_reg = LogisticRegression(
    max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
)
log_reg.fit(X_train, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)

if HAS_XGB:
    # scale_pos_weight ~ razão entre classe majoritária e minoritária no treino
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    xgb = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    xgb.fit(X_train, y_train)

modelos = {'Regressão Logística': log_reg, 'Random Forest': rf}
if HAS_XGB:
    modelos['XGBoost'] = xgb


## 5. Avaliação

Como só 0,17% das transações são fraude, um modelo que sempre prevê "normal"
teria ~99,8% de acurácia e seria inútil. Por isso avaliamos pela classe de fraude
usando **recall** (quantas fraudes reais o modelo pegou), **precisão** (das que
ele marcou como fraude, quantas eram de fato) e **F1** (equilíbrio entre as duas),
além da curva ROC e da curva Precisão-Recall.

In [ ]:
resultados = []

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    print(f"\n===== {nome} =====")
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude'], digits=4))

    resultados.append({
        'Modelo': nome,
        'Precisão (fraude)': precision_score(y_test, y_pred),
        'Recall (fraude)': recall_score(y_test, y_pred),
        'F1 (fraude)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
    })

tabela_resultados = pd.DataFrame(resultados).set_index('Modelo').round(4)
tabela_resultados


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for nome, modelo in modelos.items():
    y_proba = modelo.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f"{nome} (AUC={roc_auc_score(y_test, y_proba):.3f})")

    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec, prec, label=nome)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('Falso Positivo (FPR)')
axes[0].set_ylabel('Verdadeiro Positivo (TPR)')
axes[0].set_title('Curva ROC')
axes[0].legend()

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisão')
axes[1].set_title('Curva Precisão-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Ajuste do limiar de decisão (threshold)

Por padrão, `predict()` usa limiar 0,5. Em fraude, muitas vezes vale reduzir o
limiar para aumentar o recall (pegar mais fraudes), aceitando mais falsos
positivos — a decisão de negócio depende do custo de cada tipo de erro.

In [ ]:
melhor_modelo_nome = tabela_resultados['F1 (fraude)'].idxmax()
melhor_modelo = modelos[melhor_modelo_nome]
print("Melhor modelo pelo F1 da classe fraude:", melhor_modelo_nome)

y_proba = melhor_modelo.predict_proba(X_test)[:, 1]
prec, rec, thresholds = precision_recall_curve(y_test, y_proba)

f1_por_threshold = 2 * (prec * rec) / (prec + rec + 1e-12)
idx_melhor = np.argmax(f1_por_threshold[:-1])  # thresholds tem len(prec)-1
melhor_threshold = thresholds[idx_melhor]

print(f"Threshold escolhido: {melhor_threshold:.4f}")
print(f"Nesse ponto -> Precisão: {prec[idx_melhor]:.4f} | Recall: {rec[idx_melhor]:.4f} | F1: {f1_por_threshold[idx_melhor]:.4f}")

y_pred_ajustado = (y_proba >= melhor_threshold).astype(int)
print()
print(classification_report(y_test, y_pred_ajustado, target_names=['Normal', 'Fraude'], digits=4))


## 7. Explicabilidade

Primeiro olhamos a importância "nativa" das variáveis (Random Forest / XGBoost) e,
em seguida, usamos **SHAP** para entender o que pesou em cada decisão individual do
melhor modelo.

In [ ]:
if melhor_modelo_nome in ('Random Forest', 'XGBoost'):
    importancias = pd.Series(
        melhor_modelo.feature_importances_, index=X_train.columns
    ).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.barplot(x=importancias.values, y=importancias.index, ax=ax, color='teal')
    ax.set_title(f'Top 15 variáveis mais importantes — {melhor_modelo_nome}')
    ax.set_xlabel('Importância')
    plt.tight_layout()
    plt.show()
else:
    coefs = pd.Series(melhor_modelo.coef_[0], index=X_train.columns).sort_values()
    print(coefs.tail(10))
    print(coefs.head(10))


In [ ]:
# SHAP: explica a saída do modelo em termos da contribuição de cada variável.
# Para bases grandes, usamos uma amostra do conjunto de teste (mais rápido).
try:
    import shap

    amostra = X_test.sample(n=min(2000, len(X_test)), random_state=RANDOM_STATE)

    if melhor_modelo_nome in ('Random Forest', 'XGBoost'):
        explainer = shap.TreeExplainer(melhor_modelo)
        shap_values = explainer.shap_values(amostra)
        # Para RandomForest binário, shap_values pode vir como lista [classe0, classe1]
        valores_plot = shap_values[1] if isinstance(shap_values, list) else shap_values
    else:
        explainer = shap.LinearExplainer(melhor_modelo, X_train)
        valores_plot = explainer.shap_values(amostra)

    shap.summary_plot(valores_plot, amostra, show=True)
except ImportError:
    print("SHAP não está instalado. Rode: pip install shap")


## 8. Bônus — Undersampling vs. Oversampling

Como alternativa ao `class_weight`, comparamos rapidamente **undersampling**
(reduzir a classe majoritária) e **oversampling simples** (repetir exemplos da
classe minoritária) no treino da Regressão Logística, olhando o recall da fraude.

In [ ]:
def undersample(X, y, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    idx_fraude = y[y == 1].index
    idx_normal = y[y == 0].index
    idx_normal_amostrado = rng.choice(idx_normal, size=len(idx_fraude), replace=False)
    idx_final = np.concatenate([idx_fraude, idx_normal_amostrado])
    rng.shuffle(idx_final)
    return X.loc[idx_final], y.loc[idx_final]

def oversample(X, y, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    idx_fraude = y[y == 1].index
    idx_normal = y[y == 0].index
    idx_fraude_repetido = rng.choice(idx_fraude, size=len(idx_normal), replace=True)
    idx_final = np.concatenate([idx_normal, idx_fraude_repetido])
    rng.shuffle(idx_final)
    return X.loc[idx_final], y.loc[idx_final]

X_under, y_under = undersample(X_train, y_train)
X_over, y_over = oversample(X_train, y_train)

modelo_under = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE).fit(X_under, y_under)
modelo_over = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE).fit(X_over, y_over)

for nome, modelo in [('Undersampling', modelo_under), ('Oversampling', modelo_over)]:
    y_pred = modelo.predict(X_test)
    print(f"{nome}: precisão={precision_score(y_test, y_pred):.4f} "
          f"recall={recall_score(y_test, y_pred):.4f} f1={f1_score(y_test, y_pred):.4f}")


## 9. Conclusão

- A acurácia não diz nada de útil aqui: com 99,8% das transações normais, prever
  sempre "normal" já daria acurácia altíssima e recall zero para fraude.
- Recall, precisão e F1 da classe fraude (e a curva Precisão-Recall) são as
  métricas que realmente importam para decidir se o modelo é bom.
- O ajuste do limiar de decisão é uma alavanca de negócio: baixar o limiar
  aumenta o recall (menos fraude escapa) às custas de mais falsos positivos.
- O SHAP mostra quais variáveis mais empurraram cada transação para "fraude",
  dando explicabilidade a uma decisão que, de outra forma, seria uma caixa-preta.

Preencha esta seção com suas próprias observações depois de rodar o notebook —
compare os números que você obteve com os discutidos aqui e registre o que mudou
em relação ao caminho da Expert.